# Revision R1 — Mixed Leave-One-Out (A1 + A2 + A3)

Leave-one-out mixed training (N-1 datasets -> held-out test) at 10 seeds, adding MiniLM+LR-NoWeight, BERT-Weighted (A1) and DeBERTa-v3 + ModernBERT (A3). Held-outs exclude dan and jackhhao (shared positives).

**Revision R1 (KNOSYS-D-26-08416).** Results are written to `results/revision_R1/` on Drive (resume-safe across sessions). The original `results/v2/` is never modified.

## 0. Colab bootstrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Location of this repository on your Google Drive; edit if you cloned it elsewhere.
CANDIDATES = [
    '/content/drive/MyDrive/llm-jailbreak-cross-dataset-benchmark',
]
DRIVE_PROJECT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DRIVE_PROJECT, f'Project not found in {CANDIDATES}'
LOCAL_PROJECT = '/content/llm-jailbreak-cross-dataset-benchmark'
DRIVE_RESULTS_R1 = f'{DRIVE_PROJECT}/results/revision_R1'
print('Drive project :', DRIVE_PROJECT)

In [ ]:
import shutil
# Fresh local copy of the code each session (fast local disk).
if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)
shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)
os.chdir(LOCAL_PROJECT)
print('Working dir:', os.getcwd())

In [ ]:
# ModernBERT needs transformers>=4.48; DeBERTa-v3 needs sentencepiece.
!pip install -q -U "transformers>=4.48" sentencepiece
!pip install -q torch sentence-transformers scikit-learn scipy pandas numpy xgboost

In [ ]:
import torch, transformers
print('transformers:', transformers.__version__)
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Build the 7 new-seed manifests
Existing seeds 42/123/456 are deterministic and already on Drive; we build only the new seeds. (Rebuilt each session — idempotent.)

In [ ]:
NEW_SEEDS = '789 1011 1213 1415 1617 1819 2021'
!python experiments/build_manifests.py --seeds {NEW_SEEDS}

## 2. Seed `results/revision_R1/` from v2
Copies the existing 3-seed v2 results so their resume keys are already present — those cells are skipped, only new seeds / new methods run. Copies only if the target is missing (safe to re-run).

In [ ]:
import os, shutil
os.makedirs(DRIVE_RESULTS_R1, exist_ok=True)
FILES = ['mixed.csv']
for f in FILES:
    src = f'{DRIVE_PROJECT}/results/v2/{f}'
    dst = f'{DRIVE_RESULTS_R1}/{f}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst); print('seeded', f)
    else:
        print('kept  ', f, '(exists or no v2 source)')

## 3. Run

### 3a. Cheap methods (TF-IDF, BERT, BERT-Weighted, embeddings)

In [ ]:
!python experiments/runners/run_mixed.py \
  --result_file {DRIVE_RESULTS_R1}/mixed.csv \
  --methods BERT BERT-Weighted TF-IDF+LR TF-IDF+SVM --seeds 42 123 456 789 1011 1213 1415 1617 1819 2021

In [ ]:
!python experiments/runners/run_embedding.py --protocol mixed \
  --result_file {DRIVE_RESULTS_R1}/mixed.csv \
  --methods MiniLM+LR MiniLM+LR-NoWeight MiniLM+XGBoost MiniLM+XGBoost-Bal Arctic+RF BERT-frozen+LR --seeds 42 123 456 789 1011 1213 1415 1617 1819 2021

### 3b. DeBERTa-v3 (heavy — own session OK)

In [ ]:
#!python experiments/runners/run_mixed.py \
#  --result_file {DRIVE_RESULTS_R1}/mixed.csv \
#  --methods DeBERTa-v3 --seeds 42 123 456 789 1011 1213 1415 1617 1819 2021

### 3c. ModernBERT (heavy — own session OK)

In [ ]:
!python experiments/runners/run_mixed.py \
  --result_file {DRIVE_RESULTS_R1}/mixed.csv \
  --methods ModernBERT --seeds 42 123 456 789 1011 1213 1415 1617 1819 2021

## 4. Sanity check the written results

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob(f'{DRIVE_RESULTS_R1}/*.csv')):
    d = pd.read_csv(f)
    print(os.path.basename(f), '| rows:', len(d),
          '| methods:', sorted(d['Method'].unique()),
          '| seeds:', sorted(d['Seed'].unique()))